In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1562_Sri_Aurobindo_Marg_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,208.55,319.22,6.18,37.26,24.86,30.62,11.36,0.90,7.62,...,NaN,11.06,77.31,0.94,173.22,0.0,0.0,70.32,986.42,NaN
1,2024-01-02,200.53,307.67,9.87,33.07,25.63,29.02,11.54,0.79,6.57,...,NaN,9.93,76.67,0.92,172.11,0.0,0.0,78.73,986.17,NaN
2,2024-01-03,196.96,324.67,11.38,37.70,29.34,33.88,10.73,1.15,5.81,...,NaN,9.77,86.79,0.84,189.18,0.0,0.0,59.42,986.14,NaN
3,2024-01-04,227.17,365.00,14.73,36.82,31.58,40.87,10.33,1.08,4.26,...,NaN,9.93,88.27,1.68,170.25,0.0,0.0,29.55,986.34,NaN
4,2024-01-05,183.98,302.33,16.17,49.69,39.57,47.28,9.92,1.11,4.23,...,NaN,11.26,87.61,1.26,179.94,0.0,0.0,31.10,986.38,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,145.08,189.85,46.11,47.67,62.84,44.08,37.75,1.69,12.66,...,NaN,16.28,83.38,1.37,148.33,0.0,0.0,12.61,989.18,NaN
362,2024-12-28,66.11,100.48,21.77,40.25,39.11,32.84,36.15,1.25,7.30,...,NaN,16.56,83.09,1.17,110.70,0.0,0.0,20.71,989.81,NaN
363,2024-12-29,73.50,122.11,6.59,22.66,17.41,31.66,35.40,0.96,26.25,...,NaN,16.32,81.18,2.55,74.52,0.0,0.0,66.99,989.61,NaN
364,2024-12-30,73.36,125.75,10.47,19.22,18.73,26.77,35.46,1.04,30.07,...,NaN,14.53,80.75,2.03,78.65,0.0,0.0,63.16,989.48,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         208.55        319.22        6.18        37.26   
1  2024-01-02         200.53        307.67        9.87        33.07   
2  2024-01-03         196.96        324.67       11.38        37.70   
3  2024-01-04         227.17        365.00       14.73        36.82   
4  2024-01-05         183.98        302.33       16.17        49.69   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      24.86        30.62        11.36        0.90           7.62   
1      25.63        29.02        11.54        0.79           6.57   
2      29.34        33.88        10.73        1.15           5.81   
3      31.58        40.87        10.33        1.08           4.26   
4      39.57        47.28         9.92        1.11           4.23   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.79             5.05    11.06   77.31      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,2.638713,1.783483,-0.901337,0.337735,-0.423136,0.032873,-0.666938,-0.318760,-1.328923,2.676690,1.469083,-1.658822,0.651829,-1.299942,1.605216,0.0,0.0,-1.229030,0.855158
1,2024-01-02,2.482418,1.644529,-0.531755,0.005863,-0.365302,-0.166326,-0.631787,-0.687321,-1.390486,1.644700,1.366635,-1.799741,0.607423,-1.355720,1.569520,0.0,0.0,-1.079868,0.771736
2,2024-01-03,2.412846,1.849050,-0.380517,0.372585,-0.086645,0.438743,-0.789966,0.518878,-1.435046,-0.161284,2.065693,-1.819695,1.309584,-1.578828,2.118470,0.0,0.0,-1.422356,0.761725
3,2024-01-04,3.001581,2.334245,-0.044989,0.302884,0.081601,1.308998,-0.868079,0.284339,-1.525926,2.779889,2.216352,-1.799741,1.412271,0.763811,1.509705,0.0,0.0,-1.952139,0.828463
4,2024-01-05,2.159890,1.580285,0.099238,1.322262,0.681727,2.107042,-0.948145,0.384856,-1.527685,2.779889,2.722567,-1.633881,1.366478,-0.407508,1.821323,0.0,0.0,-1.924647,0.841810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.401803,0.227080,-0.192221,1.162266,2.429529,1.708642,-0.088902,2.328175,-1.033418,0.561110,0.739893,-1.007850,1.072986,-0.100734,0.804785,0.0,0.0,-2.252592,1.776142
362,2024-12-28,-0.137173,-0.848098,0.660121,0.574560,0.647177,0.309263,-0.088902,0.853933,-1.347685,-0.419281,-0.127904,-0.972932,1.052865,-0.658505,-0.405350,0.0,0.0,-2.108928,1.986367
363,2024-12-29,0.006844,-0.587875,-0.860272,-0.818669,-0.982704,0.162353,-0.088902,-0.117727,-0.236612,-0.728879,-0.826962,-1.002862,0.920343,-0.100734,-1.568854,0.0,0.0,-1.288092,1.919629
364,2024-12-30,0.004116,-0.544084,-0.471661,-1.091137,-0.883559,-0.446451,-0.088902,0.150317,-0.012638,-0.883677,-1.266887,-1.226088,0.890508,1.739911,-1.436039,0.0,0.0,-1.356022,1.876249


In [10]:
df.to_excel('sriAurobindo2024.xlsx', index=False)